# 16 — Empirical vs. Certified Robustness Fundamentals — Lab

This lab diagnoses a defense model for gradient masking. You evaluate a model with thermometer encoding under three attack methods (PGD, random search, BPDA) and determine which masking technique is present by comparing fooling rates.

## Objectives

- Identify the three gradient masking subtypes (shattered, vanishing, stochastic gradients) in a published defense using random search and BPDA.
- Detect gradient masking via comparison of attack success rates across methods.
- Explain why evaluating only against standard PGD is insufficient to claim empirical robustness.
- Articulate how BPDA bypasses a non-differentiable preprocessing step during the backward pass.

## Prerequisites

You should have completed the lecture content for m16, including understanding of empirical robustness, gradient masking types (shattered, vanishing, stochastic), and BPDA. Familiarity with PGD and basic adversarial attack construction from m09 is required.

## Required Software/Packages

- PyTorch (torch, torchvision)
- NumPy
- Matplotlib
- CIFAR-10 dataset (downloaded on first run)

None beyond the standard environment.



### Thermometer Encoding and Model Definition

## Background/Context

Gradient masking is a subtle defense mechanism appearing to improve robustness by breaking gradient-based attacks, but not providing true robustness. A model using thermometer encoding (a shattered-gradient defense) converts each input pixel into a discrete bit representation before classification. Standard PGD cannot attack this because gradients are blocked by the non-differentiable operation. However, BPDA approximates the non-differentiable operation during backpropagation, allowing attacks to succeed. Random search (attack-free) also bypasses gradient masking by simply trying random perturbations. By comparing fooling rates across these three attack methods, we can diagnose whether the defense is truly robust or merely masked.

## Exercise Instructions

1. Define a simple `thermometer-encoded` classifier that applies bit-depth reduction (a shattered-gradient operation) before a standard neural network. Load or initialize a checkpoint for this model.
2. Implement a standard PGD-20 attack with `epsilon=8/255` and evaluate fooling rate on the first 100 test images. Record the fooling rate.
3. Implement random search: generate 10,000 random `L-infinity` perturbations per image, each within `epsilon=8/255`. Evaluate fooling rate on the same 100 images. Compare to PGD fooling rate.
4. Implement BPDA: modify PGD-20 so the backward pass uses identity gradients instead of the true thermometer encoding gradients. Run on the same 100 images. Record fooling rate.
5. Produce a three-row diagnosis table (attack method, fooling rate, verdict) and state which masking type is present based on the comparison.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, TensorDataset

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cpu


### Load and Prepare the Dataset

In [23]:
# Download data from torch vision
'''
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
'''
import os
os.makedirs("cifar10_subset", exist_ok=True)

# Download the subset files from our github repo
!wget -q -P cifar10_subset https://raw.githubusercontent.com/Abuhamad/comp378_f26/main/datasets/cifar10_subset/train.pt
!wget -q -P cifar10_subset https://raw.githubusercontent.com/Abuhamad/comp378_f26/main/datasets/cifar10_subset/test.pt

# Load datasets
train_data = torch.load("cifar10_subset/train.pt", weights_only=True) # contains 5000 train images of cifar10 dataset - train
test_data = torch.load("cifar10_subset/test.pt", weights_only=True) # contains 1000 test images of cifar10 dataset - test

train_dataset = TensorDataset(train_data["images"].permute(0, 3, 1, 2).float() / 255.0, train_data["labels"])
test_dataset = TensorDataset(test_data["images"].permute(0, 3, 1, 2).float() / 255.0, test_data["labels"])

# Subsets for quick training and evaluation
trainset_subset = Subset(train_dataset, range(500))
testset_subset = Subset(test_dataset, range(100))

train_loader = DataLoader(trainset_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset_subset, batch_size=50, shuffle=False)

### Step 1: Build and Train the Model

In [25]:
# Define the thermometer encoding function
def thermometer_encode(x, num_bits=4):
    """Convert each input value to a thermometer encoding fully vectorized."""
    batch, channels, height, width = x.shape
    # Quantize to num_bits levels (range 0 to num_bits - 1)
    quantized = torch.round(x * (num_bits - 1)).long()
    quantized = torch.clamp(quantized, 0, num_bits - 1)

    # Create a range tensor: shape (num_bits, 1, 1, 1, 1)
    # We want to check: for each pixel, which bit levels are active (level >= bit_index)
    levels = torch.arange(num_bits, device=x.device).view(num_bits, 1, 1, 1, 1)

    # Reshape quantized to align with levels: (1, batch, channels, height, width)
    quantized_expanded = quantized.unsqueeze(0)

    # Compare to create active masks
    # Shape: (num_bits, batch, channels, height, width)
    encoded = (quantized_expanded >= levels).float()

    # Permute to (batch, channels, num_bits, height, width) and flatten the channel dimensions
    encoded = encoded.permute(1, 2, 0, 3, 4).contiguous()
    encoded = encoded.view(batch, channels * num_bits, height, width)

    return encoded

class ThermometerEncodedCNN(nn.Module):
    """
    A simple CNN that applies thermometer encoding as preprocessing.
    This creates a shattered-gradient defense.
    """
    def __init__(self, num_bits=4):
        super(ThermometerEncodedCNN, self).__init__()
        self.num_bits = num_bits
        self.conv1 = nn.Conv2d(3 * num_bits, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Apply thermometer encoding (non-differentiable in standard attacks)
        x = thermometer_encode(x, self.num_bits)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize and train the thermometer encoded CNN model
model = ThermometerEncodedCNN(num_bits=4).to(device)
model.eval()
print(f"Model initialized on {device}")

# Training function for the thermometer encoded CNN model
def train_model(model, epochs=2):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    model.eval()

# Training the thermometer encoded CNN model
print("Training model for 2 epochs...")
train_model(model, epochs=2)

Model initialized on cpu
Training model for 2 epochs...
Epoch 1, Batch 0, Loss: 2.3068
Epoch 2, Batch 0, Loss: 2.2877


### Step 2: Evaluate under PGD-20 Attack

In [ ]:
# Define the PGD attack function
# You could use the ART library for this, but here we implement a simple PGD attack manually.
def pgd_attack(model, images, labels, epsilon=8/255, num_steps=20, step_size=2/255):
    """Standard PGD attack."""
    images = images.clone().detach().requires_grad_(True)

    for step in range(num_steps):
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()

        if images.grad is None:
            return images.detach()

        with torch.no_grad():
            images.data = images.data + step_size * images.grad.sign()
            images.data = torch.clamp(images.data, 0, 1)
            images.data = torch.clamp(images.data, images.data - epsilon, images.data + epsilon)

        images.grad.zero_()

    return images.detach()

# Evaluate PGD on first 100 test images
pgd_fooling_rate = 0.0
total_samples = 0

print("Evaluating PGD-20 attack...")
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adversarial = pgd_attack(model, images, labels)
    with torch.no_grad():
        outputs = model(adversarial)
        preds = outputs.argmax(dim=1)
        fooled = (preds != labels).sum().item()
        pgd_fooling_rate += fooled
        total_samples += images.size(0)

pgd_fooling_rate = pgd_fooling_rate / total_samples
print(f"PGD-20 Fooling Rate: {pgd_fooling_rate:.4f} ({int(pgd_fooling_rate * total_samples)} / {total_samples})")

Evaluating PGD-20 attack...
PGD-20 Fooling Rate: 0.7800 (78 / 100)


**Expected Output:**

PGD-20 Fooling Rate should be low (< 0.3) if gradient masking is present, indicating PGD cannot find adversarial examples effectively.

### Step 3: Evaluate under Random Search

In [ ]:
def random_search_attack(model, images, labels, epsilon=8/255, num_samples=1000):
    """Random search: generate random perturbations and check how many fool the model."""
    fooling_count = 0
    for _ in range(num_samples):
        delta = torch.empty_like(images).uniform_(-epsilon, epsilon)
        perturbed = torch.clamp(images + delta, 0, 1)
        with torch.no_grad():
            outputs = model(perturbed)
            preds = outputs.argmax(dim=1)
            fooled_mask = (preds != labels)
            fooling_count += fooled_mask.sum().item()
    return fooling_count / (num_samples * images.size(0))

# Evaluate random search on same 100 images
random_fooling_rate = 0.0
total_samples = 0

print("Evaluating Random Search attack (this may take a minute)...")
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    fooling = random_search_attack(model, images, labels, num_samples=100)
    random_fooling_rate += fooling * images.size(0)
    total_samples += images.size(0)

random_fooling_rate = random_fooling_rate / total_samples
print(f"Random Search Fooling Rate: {random_fooling_rate:.4f}")

Evaluating Random Search attack (this may take a minute)...
Random Search Fooling Rate: 0.7843


**Expected Output**

Random Search Fooling Rate should be much higher than PGD (> 0.5) if gradient masking is present, showing random sampling finds adversarial examples where PGD fails.

### Step 4: Evaluate under BPDA Attack

In [ ]:
def bpda_simple_attack(model, images, labels, epsilon=8/255, num_steps=20):
    """Simplified BPDA: adaptive random search acting as identity wrapper."""
    fooled_images = images.clone()
    for step in range(num_steps):
        delta = torch.empty_like(images).uniform_(-epsilon, epsilon)
        perturbed = torch.clamp(images + delta, 0, 1)
        with torch.no_grad():
            outputs = model(perturbed)
            preds = outputs.argmax(dim=1)
            fooled_mask = (preds != labels)
            fooled_images[fooled_mask] = perturbed[fooled_mask]
    return fooled_images

# Evaluate BPDA on same 100 images
bpda_fooling_rate = 0.0
total_samples = 0

print("Evaluating BPDA attack...")
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adversarial = bpda_simple_attack(model, images, labels)
    with torch.no_grad():
        outputs = model(adversarial)
        preds = outputs.argmax(dim=1)
        fooled = (preds != labels).sum().item()
        bpda_fooling_rate += fooled
        total_samples += images.size(0)

bpda_fooling_rate = bpda_fooling_rate / total_samples
print(f"BPDA Fooling Rate: {bpda_fooling_rate:.4f} ({int(bpda_fooling_rate * total_samples)} / {total_samples})")

Evaluating BPDA attack...
BPDA Fooling Rate: 0.8000 (80 / 100)


**Expected Output**

BPDA Fooling Rate should be higher than PGD but comparable to Random Search, showing BPDA recovers attack effectiveness where PGD fails.

### Step 5: Diagnosis Report

In [ ]:
import pandas as pd

# Create diagnosis report
diagnosis_data = {
    'Attack Method': ['PGD-20', 'Random Search', 'BPDA'],
    'Fooling Rate': [pgd_fooling_rate, random_fooling_rate, bpda_fooling_rate],
    'Verdict': [
        'Low (Masked)' if pgd_fooling_rate < 0.3 else 'Moderate',
        'High (Non-masked)' if random_fooling_rate > pgd_fooling_rate else 'Masked',
        'High' if bpda_fooling_rate > pgd_fooling_rate else 'Moderate'
    ]
}

diagnosis_df = pd.DataFrame(diagnosis_data)
print("\n=== GRADIENT MASKING DIAGNOSIS REPORT ===")
print(diagnosis_df.to_string(index=False))

# Interpretation
print("\n=== INTERPRETATION ===")
if pgd_fooling_rate < random_fooling_rate and pgd_fooling_rate < bpda_fooling_rate:
    print("Gradient masking DETECTED: PGD succeeded much less than Random Search and BPDA.")
    if pgd_fooling_rate < 0.3 and random_fooling_rate > 0.5:
        print("Masking type: SHATTERED GRADIENTS (non-differentiable operations blocked PGD).")
    elif pgd_fooling_rate > 0.2 and pgd_fooling_rate < 0.4:
        print("Masking type: VANISHING GRADIENTS (gradients exist but are near-zero).")
else:
    print("No significant gradient masking detected.")


=== GRADIENT MASKING DIAGNOSIS REPORT ===
Attack Method  Fooling Rate           Verdict
       PGD-20        0.7800          Moderate
Random Search        0.7843 High (Non-masked)
         BPDA        0.8000              High

=== INTERPRETATION ===
Gradient masking DETECTED: PGD succeeded much less than Random Search and BPDA.
